# CODIGO 1 - FILTRACION AUTOMATIZADA

Este notebook reemplaza el CODIGO 1 original.

Cambios respecto a la version anterior:

1. NO se filtra por grupo taxonomico de entrada. Se caracteriza / procesa
   toda la base (taxonomy + train) sin descartar ningun grupo (Aves, Amphibia,
   Insecta, Mammalia, etc).
2. Se agrega la duracion del audio en **muestras exactas** (`num_muestras`)
   y la **frecuencia de muestreo** (`frecuencia_muestreo`) de cada audio,
   en lugar de solo segundos. Esto porque los segundos no son exactos y
   porque los audios pueden venir de dispositivos con distintas tasas de
   muestreo.
3. El filtro de duracion (10 segundos) se calcula con muestras:
   `num_muestras <= frecuencia_muestreo * 10`.
4. El filtro por grupo taxonomico queda automatizado: una funcion recibe
   el nombre del grupo ("Aves", "Amphibia", "Insecta") y devuelve el csv filtrado.
5. Todo se hace por posicion (`.iloc`, `df.columns[indice]`), tal como se ha
   venido trabajando en el resto del proyecto

El notebook esta dividido en celdas porque la celda que mide duracion de
audio (ETAPA 3) es lenta (recorre archivo por archivo) y conviene poder
ejecutarla aparte del resto.

In [2]:
import os
import pandas as pd
import soundfile as sf

RUTA_TAXONOMY = r"C:\Users\manue\Downloads\3ccbe\DATOS\taxonomy.csv"
RUTA_TRAIN = r"C:\Users\manue\Downloads\3ccbe\DATOS\train.csv"
CARPETA_AUDIOS = r"C:\Users\manue\Downloads\3ccbe\train_audio"
SALIDA_COMPLETO = r"C:\Users\manue\Downloads\3ccbe\Proyecto_final\Filtracion 2\Audios.csv"
CARPETA_SALIDA_GRUPOS = r"C:\Users\manue\Downloads\3ccbe\Proyecto_final\Filtracion 2"

taxonomy = pd.read_csv(RUTA_TAXONOMY)
train = pd.read_csv(RUTA_TRAIN)
print(f"Registros taxonomy : {len(taxonomy)}")
print(f"Registros train    : {len(train)}")
print(f"Columnas taxonomy  : {list(taxonomy.columns)}")
print(f"Columnas train     : {list(train.columns)}")


Registros taxonomy : 206
Registros train    : 28564
Columnas taxonomy  : ['primary_label', 'inat_taxon_id', 'scientific_name', 'common_name', 'class_name']
Columnas train     : ['primary_label', 'secondary_labels', 'type', 'filename', 'collection', 'rating', 'url', 'latitude', 'longitude', 'scientific_name', 'common_name', 'author', 'license']


In [3]:
train_unido = train.merge(taxonomy,left_on=[
        train.columns[0],
        train.columns[9],
        train.columns[10]],
    right_on=[
        taxonomy.columns[0],
        taxonomy.columns[2],
        taxonomy.columns[3]
    ],how="inner")

print(f"Registros taxonomy : {len(taxonomy)}")
print(f"Registros train    : {len(train)}")
print(f"Registros unidos   : {len(train_unido)}")

Registros taxonomy : 206
Registros train    : 28564
Registros unidos   : 28564


En vez de volver a ejecutar todo ese bloque que lee los 28mil audios, y que ya lo habia ejecutado... se hace

In [ ]:
df_completo = pd.read_csv(r"C:\Users\manue\Downloads\3ccbe\Proyecto_final\Filtracion 2\Audios.csv")

Esta celda recorre cada archivo de audio con `soundfile` y agrega dos
columnas nuevas usando el numero exacto de muestras (`frames`) y la
frecuencia de muestreo real de cada archivo, en lugar de la duracion en
segundos.

In [ ]:
registros_validos = []
audios_revisados = 0
audios_no_encontrados = 0
audios_error_lectura = 0

for fila in train_unido.itertuples(index=False, name=None):
    ruta_audio = os.path.join(CARPETA_AUDIOS,fila[3])
    audios_revisados += 1

    if not os.path.exists(ruta_audio):
        audios_no_encontrados += 1
        continue

    try:
        info = sf.info(ruta_audio)

    except Exception:
        audios_error_lectura += 1
        continue

    registro = dict(zip(train_unido.columns, fila))
    registro["ruta_audio"] = ruta_audio
    registro["num_muestras"] = info.frames
    registro["frecuencia_muestreo"] = info.samplerate
    registros_validos.append(registro)

    if audios_revisados % 1000 == 0:
        print(f"Audios revisados : {audios_revisados}")

df_completo = pd.DataFrame(registros_validos)

print(f"Audios revisados        : {audios_revisados}")
print(f"Audios no encontrados   : {audios_no_encontrados}")
print(f"Audios con error        : {audios_error_lectura}")
print(f"Registros finales       : {len(df_completo)}")

In [5]:
print(df_completo.columns)
print(df_completo.iloc[:,14].value_counts())
df_completo.to_csv(SALIDA_COMPLETO, index=False)
print(SALIDA_COMPLETO)


Index(['primary_label', 'secondary_labels', 'type', 'filename', 'collection',
       'rating', 'url', 'latitude', 'longitude', 'scientific_name',
       'common_name', 'author', 'license', 'inat_taxon_id', 'class_name',
       'ruta_audio', 'num_muestras', 'frecuencia_muestreo'],
      dtype='str')
class_name
Aves        27648
Amphibia      583
Mammalia      178
Insecta       155
Name: count, dtype: int64
C:\Users\manue\Downloads\3ccbe\Proyecto_final\Filtracion 2\Audios.csv


`filtrar_por_duracion_y_grupo(df, nombre_grupo, segundos_max)` recibe
cualquier nombre de grupo taxonomico presente en la columna de clase y el
limite de segundos, y devuelve el subconjunto filtrado. El limite de
segundos se convierte a muestras usando la frecuencia de muestreo de cada
fila (no todas las filas tienen la misma), por eso no se puede comparar
un solo numero fijo de muestras para toda la base.

In [6]:
def filtrar_por_duracion_y_grupo(df, nombre_grupo, segundos_max=5):
    grupo_valido = df.iloc[:, 14] == nombre_grupo
    limite_muestras = df.iloc[:, 17] * segundos_max
    duracion_valida = df.iloc[:, 16] <= limite_muestras
    return df[grupo_valido & duracion_valida].copy()

GRUPOS_A_GENERAR = ["Aves","Amphibia","Insecta","Mammalia",]

for grupo in GRUPOS_A_GENERAR:
    df_grupo = filtrar_por_duracion_y_grupo(df_completo,grupo,segundos_max=5)
    ruta_salida = os.path.join(CARPETA_SALIDA_GRUPOS,f"{grupo}_5S.csv")
    df_grupo.to_csv(ruta_salida, index=False)

    print("\n" + "=" * 60)
    print(f"Grupo: {grupo}")
    print(f"Registros: {len(df_grupo)}")
    print(f"Archivo generado: {ruta_salida}")



Grupo: Aves
Registros: 2256
Archivo generado: C:\Users\manue\Downloads\3ccbe\Proyecto_final\Filtracion 2\Aves_5S.csv

Grupo: Amphibia
Registros: 45
Archivo generado: C:\Users\manue\Downloads\3ccbe\Proyecto_final\Filtracion 2\Amphibia_5S.csv

Grupo: Insecta
Registros: 11
Archivo generado: C:\Users\manue\Downloads\3ccbe\Proyecto_final\Filtracion 2\Insecta_5S.csv

Grupo: Mammalia
Registros: 9
Archivo generado: C:\Users\manue\Downloads\3ccbe\Proyecto_final\Filtracion 2\Mammalia_5S.csv


In [7]:
import glob

archivos_csv = glob.glob(os.path.join(CARPETA_SALIDA_GRUPOS, "*.csv"))

df_unido = pd.concat([pd.read_csv(f) for f in archivos_csv], ignore_index=True)

ruta_final = os.path.join(CARPETA_SALIDA_GRUPOS, "dataset_completo_5S.csv")
df_unido.to_csv(ruta_final, index=False)

print(f"CSV unidos: {len(archivos_csv)}")
print(f"Total registros: {len(df_unido)}")
print(f"Guardado en: {ruta_final}")

CSV unidos: 9
Total registros: 46200
Guardado en: C:\Users\manue\Downloads\3ccbe\Proyecto_final\Filtracion 2\dataset_completo_5S.csv
